In [39]:
print("Please upload a PNG file using the widget below.")

from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', mutiple=False)
display(uploader)

Please upload a PNG file using the widget below.


FileUpload(value=(), accept='.png', description='Upload')

In [41]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Locate .env in your project directory
env_path = Path.cwd() / ".env"

print("Current directory:", Path.cwd())
print("Env file exists:", env_path.exists())

# Load and overwrite existing environment variables
load_dotenv(env_path, override=True)

key = os.getenv("OPENAI_API_KEY")

print("Key loaded:", bool(key))
print("Is placeholder:", not key or "your_openai_api_key_here" in key)

Current directory: /Users/uditbhatt/langchain/Automatic Chef
Env file exists: True
Key loaded: True
Is placeholder: False


In [42]:
import base64
uploaded_file = uploader.value[0]
content_mv = uploaded_file['content']
img_bytes = bytes(content_mv)
img_b64 = base64.b64encode(img_bytes).decode('utf-8')

In [43]:
from langchain_core.tools import tool
from tavily import TavilyClient
tavily_client = TavilyClient()
@tool(
    "Recepie_Search",
    description="Search web for recepies based on the left over items left in the fridge from the image"
)
def recepie_search(image_b64: str) -> str:
    """
    Search web for recepies based on the left over items left in the fridge from the image
    """
    prompt = f"Search web for recepies based on the left over items left in the fridge from the image: {image_b64}"
    response = tavily_client.generate_text(prompt)
    return response

In [44]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv(override=True)

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0
)

Recepie_agent = create_agent(
    model=llm,
    tools=[recepie_search],
    checkpointer=InMemorySaver(),
)

In [45]:
from langchain.messages import HumanMessage
question = HumanMessage(content=f"Search web for recepies based on the left over items left in the fridge from the image: {img_b64}")
config = {"configurable" : {"thread_id" : "1"}}

In [ ]:
load_dotenv()
response = Recepie_agent.invoke(
    {"messages" : [question]},
    config
)